# Gemma 3 核心架构复现与官方权重对齐

目标：把 `google/gemma-3-1b-it` 官方权重载入独立 PyTorch 实现，比较局部注意力层、全局注意力层和最终 logits。运行前请选择 GPU，并在 Hugging Face 接受模型许可；在 Colab Secrets 添加 `HF_TOKEN`，不要把 Token 写入代码或提交到 GitHub。

In [ ]:
!nvidia-smi
REPO_URL = 'https://github.com/zmh2245749337/gemma-workorder.git'
!git clone {REPO_URL}
%cd gemma-workorder
!pip -q install -r requirements-colab.txt
!pip -q install -e . --no-deps

In [ ]:
import os
from google.colab import userdata
token = userdata.get('HF_TOKEN')
if not token:
    raise ValueError('没有读取到 HF_TOKEN，请在 Colab Secrets 中添加。')
os.environ['HF_TOKEN'] = token

## 1. 不下载模型的结构与缓存单元测试

测试 RMSNorm、5:1 层模式、MQA 张量形状、滑动窗口 Mask，以及完整前向与 KV Cache 增量解码的一致性。

In [ ]:
!pytest -q

## 2. 官方权重数值对齐

第 0 层和第 25 层是滑动窗口注意力，第 5 层是全局注意力。脚本会保存最大/平均绝对误差、余弦相似度与 top-1 token 是否一致。

In [ ]:
!python scripts/run_core_alignment.py --precision fp16 --layers 0 5 25 --output reports/core_alignment.json

In [ ]:
import json
from pathlib import Path
report = json.loads(Path('reports/core_alignment.json').read_text(encoding='utf-8'))
report

## 3. 下载证据

跑完以后保存 JSON。只有这份报告里的真实数据，才可以写进简历。

In [ ]:
from google.colab import files
files.download('reports/core_alignment.json')